In [2]:
from collections import defaultdict
from pathlib import Path
import re
import sys

import fitz  # PyMuPDF
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.parsing import extract_layout_features, extract_text_pdfplumber

PDF_DIR = ROOT / 'data' / 'raw' / 'data' / 'data'
assert PDF_DIR.exists(), f'PDF directory not found: {PDF_DIR}'

In [ ]:
# Choose up to ten files from different resume categories.
by_category = defaultdict(list)
for pdf_path in sorted(PDF_DIR.rglob('*.pdf')):
    by_category[pdf_path.parent.name].append(pdf_path)

sample_pdfs = [paths[0] for _, paths in sorted(by_category.items())][:10]
display(pd.DataFrame({
    'category': [path.parent.name for path in sample_pdfs],
    'file': [path.name for path in sample_pdfs],
    'size_kb': [round(path.stat().st_size / 1024, 1) for path in sample_pdfs],
}))

In [ ]:
GARBLED_MARKERS = re.compile(r'\ufffd|\(cid:|[âÃ][^\s]{1,3}')

def inspect_pdf(pdf_path):
    """Extract text and identify scan / multi-column review candidates."""
    text = extract_text_pdfplumber(str(pdf_path))
    with fitz.open(pdf_path) as document:
        page_count = len(document)
        embedded_chars = sum(len(page.get_text('text').strip()) for page in document)
        multi_column = False
        for page in document:
            left_edges = sorted(block[0] for block in page.get_text('blocks') if block[4].strip())
            if any(right - left > page.rect.width * 0.22 for left, right in zip(left_edges, left_edges[1:])):
                multi_column = True

    return {
        'path': pdf_path, 'text': text,
        'is_likely_scanned': embedded_chars < 50 * max(page_count, 1),
        'multi_column_suspected': multi_column,
        'garbled_markers': len(GARBLED_MARKERS.findall(text)),
        **extract_layout_features(str(pdf_path)),
    }

records, failures = [], []
for pdf_path in sample_pdfs:
    try:
        records.append(inspect_pdf(pdf_path))
    except Exception as error:
        failures.append({'file': pdf_path.name, 'error': str(error)})

assert not failures, failures
results = pd.DataFrame(records)
results['file'] = results.path.map(lambda path: path.name)
results['category'] = results.path.map(lambda path: path.parent.name)
results['text_chars'] = results.text.str.len()
results['status'] = results.apply(
    lambda row: 'needs OCR' if row.is_likely_scanned else ('review column order' if row.multi_column_suspected else 'text extracted'), axis=1
)

In [ ]:
# Print extracted text. Look for replacement characters and scrambled column order.
for row in results.itertuples():
    display(Markdown(f'### {row.file} — {row.status}'))
    if row.is_likely_scanned:
        print('No usable embedded text: route this file to OCR and do not index an empty string.')
    else:
        print(row.text[:1200])
        if len(row.text) > 1200:
            print('… [preview truncated]')

In [ ]:
# section_count is a font-size heuristic. Review it against the previews above.
layout_columns = ['file', 'category', 'status', 'num_pages', 'text_chars', 'section_count',
                  'avg_font_size', 'max_font_size', 'font_size_variance', 'bold_ratio',
                  'garbled_markers', 'multi_column_suspected']
display(results[layout_columns].sort_values(['section_count', 'file'], ascending=[False, True]))

if (results.section_count == 0).all():
    print('All section counts are zero: font metadata does not distinguish headings in this sample. Add a text-heading fallback before relying on this feature.')
else:
    print('Review zero or unusually high section counts against the text previews; heading detection is heuristic.')

In [ ]:
# Edge-case handling report.
edge_cases = results.loc[
    results.is_likely_scanned | results.multi_column_suspected | (results.garbled_markers > 0),
    ['file', 'category', 'status', 'garbled_markers', 'text_chars', 'num_pages'],
]
display(edge_cases if not edge_cases.empty else pd.DataFrame({'result': ['No scan, multi-column, or garbled-text candidates in this sample.']}))
print('Policy: OCR scan candidates; manually verify reading order for multi-column candidates; exclude text that remains empty or garbled.')